In [ ]:
from kafka import KafkaConsumer
import json
import pyodbc

consumer = KafkaConsumer(
    'healthcare_stream',
    bootstrap_servers='localhost:9092',
    auto_offset_reset='latest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=localhost\\SQLEXPRESS;'
    'DATABASE=Health_Stream_TeleMedicine;'
    'Trusted_Connection=yes;'
)

cursor = conn.cursor()

print("Consumer started... waiting for messages")

for message in consumer:
    data = message.value

    print("Received:", data)

    try:
        cursor.execute("""
        INSERT INTO healthcare_iot_target_dataset
        (
            Patient_ID,
            Timestamp,
            Sensor_ID,
            Sensor_Type,
            Temperature_C,
            Systolic_BP_mmHg,
            Diastolic_BP_mmHg,
            Heart_Rate_bpm,
            Device_Battery_Level,
            Target_Blood_Pressure,
            Target_Heart_Rate,
            Target_Health_Status,
            Battery_Level
        )
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
        """,
        data['Patient_ID'],
        data['Timestamp'],
        data['Sensor_ID'],
        data['Sensor_Type'],
        data['Temperature_C'],
        data['Systolic_BP_mmHg'],
        data['Diastolic_BP_mmHg'],
        data['Heart_Rate_bpm'],
        data['Device_Battery_Level'],
        data['Target_Blood_Pressure'],
        data['Target_Heart_Rate'],
        data['Target_Health_Status'],
        data['Battery_Level'])

        conn.commit()

        print("Inserted into SQL Server ✔")

    except Exception as e:
        print("SQL Insert Error:", e)

Consumer started... waiting for messages
